# NLP Masterclass 05: Pre-training Objectives & BERT

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 04 (Seq2Seq & Attention), DL 07 (Transformer Architecture)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"GPT is trained to predict the NEXT token. BERT is trained to predict MASKED tokens. These are both 'self-supervised' — no human labels needed. How can predicting randomly missing words teach a model to perform sentiment analysis, question answering, and named entity recognition?"*

---

## Why Pre-training Changed Everything

Before BERT (2018), you trained a separate model for EACH task: one for sentiment, one for NER, one for QA. Each needed thousands of labeled examples.

After BERT: Train ONE model on unlabeled text (billions of words from the internet), then **fine-tune** it on ANY task with just a few hundred labeled examples.

This is the same paradigm used in NB11 (LoRA fine-tuning) and NB12 (DPO alignment).

## Table of Contents
1. Self-Supervised Learning: The Core Idea
2. Causal Language Modeling (GPT-style)
3. Masked Language Modeling (BERT-style)
4. BERT Architecture & Fine-tuning
5. Modern Pre-training: From BERT to Llama

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors see pre-training as "training on lots of data." Seniors understand that the pre-training OBJECTIVE determines what the model learns. CLM (GPT) learns generation. MLM (BERT) learns representation. The choice of objective directly determines what downstream tasks the model excels at.

**Mental Model:** Pre-training is like medical school. You don't learn ANY specific specialty — you learn anatomy, physiology, pharmacology (general knowledge). Fine-tuning is residency: specializing in cardiology (sentiment analysis) or surgery (code generation). The better your general education, the faster you specialize.

**Common Junior Pitfall:** Using GPT (decoder-only, CLM) for embedding/classification tasks where BERT (encoder-only, MLM) would be dramatically better. GPT is optimized for generation. BERT is optimized for understanding.

---

In [ ]:
!pip install -q torch transformers datasets

## 1 · Self-Supervised Learning

### Traditional Supervised Learning
```
Data: ("This movie is great!", positive)  ← Human labeled
Cost: $$$$ (need thousands of labels per task)
```

### Self-Supervised Learning (Pre-training)
```
Data: "The cat sat on the ____"  ← Label is FREE (just hide a word!)
Cost: Free! The text IS the label.
```

**Two Main Objectives:**

| Objective | Task | Model | Direction |
|-----------|------|-------|-----------|
| **CLM** (Causal LM) | Predict NEXT token | GPT, Llama | Left-to-right only |
| **MLM** (Masked LM) | Predict MASKED token | BERT | Bidirectional |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# OBJECTIVE 1: Causal Language Modeling (GPT)
# ==========================================

print('=== Causal Language Modeling (CLM) ===')
print()
print('Input:  "The    cat    sat    on"')
print('Target: "cat    sat    on     the"  (shifted by 1)')
print()

# Each position predicts the NEXT token
vocab = {'<pad>':0, 'the':1, 'cat':2, 'sat':3, 'on':4, 'mat':5}
tokens = torch.tensor([[1, 2, 3, 4, 1, 5]])  # "the cat sat on the mat"

# Input and targets for CLM
clm_input  = tokens[:, :-1]  # "the cat sat on the"
clm_target = tokens[:, 1:]   # "cat sat on the mat"

print(f'CLM Input:  {clm_input.tolist()[0]}  ("the cat sat on the")')
print(f'CLM Target: {clm_target.tolist()[0]}  ("cat sat on the mat")')
print(f'\nThis is autoregressive: each position learns to predict the next word.')
print(f'This is EXACTLY how GPT-4, Llama, and Claude are pre-trained.')

# ==========================================
# OBJECTIVE 2: Masked Language Modeling (BERT)
# ==========================================

print('\n\n=== Masked Language Modeling (MLM) ===')
print()

def apply_mlm_masking(tokens, mask_prob=0.15, mask_token_id=99):
    """Apply BERT-style MLM masking.
    
    Rule: 15% of tokens are selected for prediction.
    Of those 15%:
      - 80% are replaced with [MASK]
      - 10% are replaced with a random token
      - 10% are kept unchanged
    """
    input_ids = tokens.clone()
    labels = torch.full_like(tokens, -100)  # -100 = ignore in loss
    
    # Select ~15% of positions for masking
    mask_positions = torch.rand(tokens.shape) < mask_prob
    labels[mask_positions] = tokens[mask_positions]  # Set labels at masked positions
    
    # Apply the 80/10/10 rule
    for i in range(tokens.shape[0]):
        for j in range(tokens.shape[1]):
            if mask_positions[i, j]:
                r = torch.rand(1).item()
                if r < 0.8:
                    input_ids[i, j] = mask_token_id  # Replace with [MASK]
                elif r < 0.9:
                    input_ids[i, j] = torch.randint(1, 6, (1,)).item()  # Random token
                # else: keep unchanged (10%)
    
    return input_ids, labels

# Demo MLM
original = torch.tensor([[1, 2, 3, 4, 1, 5]])  # "the cat sat on the mat"
masked_input, mlm_labels = apply_mlm_masking(original, mask_prob=0.3)

words = {0:'<pad>', 1:'the', 2:'cat', 3:'sat', 4:'on', 5:'mat', 99:'[MASK]'}
print(f'Original:  {[words.get(t.item(), "?") for t in original[0]]}')
print(f'Masked:    {[words.get(t.item(), "?") for t in masked_input[0]]}')
print(f'Labels:    {[words.get(t.item(), "ignore") for t in mlm_labels[0]]}')
print(f'\nModel must predict the original tokens at [MASK] positions.')
print(f'Using BIDIRECTIONAL context (sees both left AND right of [MASK]).')

---
## 2 · BERT Architecture

```
[CLS] The cat [MASK] on the mat [SEP]
  ↓    ↓    ↓     ↓    ↓   ↓   ↓   ↓
  Embedding Layer + Positional Encoding
  ↓
  [Transformer Encoder Block] × 12  (Bidirectional self-attention!)
  ↓
  h_CLS  h_The  h_cat  h_MASK  h_on  h_the  h_mat  h_SEP
    ↓                    ↓
  Classification      Predict "sat"
  (fine-tuning)       (pre-training)
```

### BERT Special Tokens

| Token | Purpose |
|-------|--------|
| `[CLS]` | Classification token. Its final embedding represents the whole sentence. |
| `[SEP]` | Separator between sentence pairs. |
| `[MASK]` | Placeholder for masked tokens during MLM training. |
| `[PAD]` | Padding to make sequence lengths equal. |

In [ ]:
# Using BERT with HuggingFace
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline
import torch

# Load BERT
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForMaskedLM.from_pretrained('bert-base-uncased')

# Inspect architecture
total_params = sum(p.numel() for p in model.parameters())
print(f'BERT-base architecture:')
print(f'  Parameters: {total_params:,} ({total_params/1e6:.0f}M)')
print(f'  Layers: 12 transformer encoder blocks')
print(f'  Hidden dim: 768')
print(f'  Attention heads: 12')
print(f'  Vocabulary: {tokenizer.vocab_size:,} tokens')

# MLM in action: predict the masked word
text = 'The capital of France is [MASK].'
inputs = tokenizer(text, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Find the [MASK] position
mask_idx = (inputs['input_ids'] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
mask_logits = logits[0, mask_idx, :]

# Top 5 predictions
top5 = torch.topk(mask_logits, 5, dim=-1)
print(f'\nFill in the blank: "{text}"')
for i, (token_id, score) in enumerate(zip(top5.indices[0], top5.values[0])):
    word = tokenizer.decode(token_id)
    print(f'  {i+1}. "{word}" (score: {score:.2f})')

---
## 3 · Fine-tuning BERT (Classification)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import torch.nn as nn

# Fine-tuning concept:
# 1. Take pre-trained BERT (MLM knowledge)
# 2. Replace MLM head with classification head
# 3. Train on labeled data (freeze lower layers, train upper + head)

model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3  # positive, negative, neutral
)
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Show what changed
print('Fine-tuning for Classification:')
print(f'  BERT encoder: {sum(p.numel() for n,p in model.named_parameters() if "classifier" not in n):,} params (pre-trained)')
print(f'  Classification head: {sum(p.numel() for n,p in model.named_parameters() if "classifier" in n):,} params (random init)')

# Forward pass
texts = ['This movie was absolutely fantastic!', 'It was okay, nothing special.', 'Terrible waste of time.']
inputs = tokenizer(texts, padding=True, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

probs = torch.softmax(outputs.logits, dim=-1)
labels = ['Positive', 'Neutral', 'Negative']
print(f'\nPredictions (before fine-tuning — random classifier head):')
for text, prob in zip(texts, probs):
    pred = labels[prob.argmax()]
    print(f'  "{text[:40]}..." → {pred} ({prob.max():.2%})')
print(f'\nNote: Predictions are random because the classifier head is untrained.')
print(f'After fine-tuning on ~1000 labeled examples, accuracy would be >90%.')
print(f'This is the power of pre-training: 110M params already "understand" language.')

---
## 4 · Modern Pre-training: The Evolution

| Model | Year | Objective | Architecture | Key Innovation |
|-------|------|-----------|-------------|----------------|
| BERT | 2018 | MLM + NSP | Encoder | Bidirectional pre-training |
| GPT-2 | 2019 | CLM | Decoder | Scaled up + shown emergent abilities |
| T5 | 2020 | Span Corruption | Enc-Dec | Unified text-to-text framework |
| GPT-3 | 2020 | CLM | Decoder | 175B params, in-context learning |
| Llama 2 | 2023 | CLM | Decoder | Open-source, RoPE, GQA |
| Llama 3 | 2024 | CLM | Decoder | 15T tokens, 128K context |

### Why CLM (GPT-style) Dominated

1. **Simpler objective** — just predict next token
2. **Scales better** — more data + more params = consistently better
3. **Emergent abilities** — at scale, CLM models learn reasoning, coding, math
4. **Generation native** — CLM models generate text naturally; BERT cannot

### When to Use What (2026 Guidance)

| Task | Best Choice | Why |
|------|-------------|-----|
| Text generation / Chat | Decoder (Llama, GPT) | Native generation |
| Embeddings / Retrieval | Encoder (BERT, Sentence-BERT) | Bidirectional = better representations |
| Classification | Either (BERT for small data, GPT for prompting) | Depends on label budget |
| Translation | Encoder-Decoder (T5, NLLB) | Seq2Seq naturally fits |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How can predicting missing words teach a model sentiment analysis?*

**A:** To predict "The movie was [MASK]" correctly, the model must understand:
- Grammar (the word should be an adjective)
- Semantics (it relates to movies)
- Sentiment patterns (if preceded by "absolutely", likely positive)

MLM forces the model to learn deep linguistic representations as a SIDE EFFECT. These representations are general enough to transfer to any text task. This is why 1000 labeled sentiment examples is enough to fine-tune — the model already "understands" language.

---

## ✅ Knowledge Check

### Q1: MLM vs CLM
Why can't you use BERT for text generation (chatbots)?

<details><summary>👉 Answer</summary>

BERT uses bidirectional attention — it sees ALL tokens including future ones. In generation, future tokens don't exist yet. You can't condition on what hasn't been generated. BERT can fill in blanks (MLM) but can't generate open-ended text. GPT-style CLM only sees left context, so it naturally generates one token at a time.
</details>

### Q2: 80/10/10 rule
In MLM, why not replace ALL masked positions with [MASK]? Why the 10% random + 10% unchanged?

<details><summary>👉 Answer</summary>

If only [MASK] tokens are predicted, the model learns to ignore non-[MASK] tokens during fine-tuning (when [MASK] never appears). The 10% random replacement teaches robustness to noisy input. The 10% unchanged forces the model to build good representations for ALL tokens, not just masked ones.
</details>

### Q3: Connection to NB11
In NB11, LoRA fine-tuning uses a decoder-only model (Llama). How does LoRA relate to the pre-train → fine-tune paradigm?

<details><summary>👉 Answer</summary>

LoRA IS fine-tuning, but efficient. Traditional fine-tuning updates ALL parameters. LoRA freezes the pre-trained weights and adds small adapter matrices to the attention projections (q_proj, k_proj, v_proj). This preserves the pre-trained knowledge while adapting to the specific task with <1% of the parameters.
</details>

---

## 🔨 Practical Practice

### Exercise 1: MLM Probing
Use the BERT MLM pipeline to probe what BERT knows. Fill in: "The CEO of Apple is [MASK]." Try different companies, countries, and categories. What does BERT get right/wrong?

### Exercise 2: Feature Extraction
Use BERT as a feature extractor: tokenize 10 sentences, get the [CLS] embeddings (768-dim), compute cosine similarities between all pairs. Do similar sentences cluster?

### Exercise 3: Pre-training from Scratch
Using the MiniGPT from DL 07, implement a CLM training loop on a small text corpus. Train for 1000 steps and generate text. How does quality change with training?

---

**Next →** NLP 06: HuggingFace Pipelines & Transfer Learning